# Metadata Extraction and Vector DB Ingestion

This notebook processes PDFs, extracts metadata using Gemini, and stores them in a ChromaDB vector database.

In [ ]:
import os
import glob
import json
import random
import time
import logging
import concurrent.futures
import threading
from itertools import cycle
from dotenv import load_dotenv
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Suppress pypdf warnings
logging.getLogger("pypdf").setLevel(logging.ERROR)

# Load environment variables
load_dotenv("../.env")

# Load multiple API keys
API_KEYS = []
BASE = "GEMINI"
for i in range(3, 10):
    key_name = f"{BASE}{i}"
    if os.getenv(key_name):
        API_KEYS.append(os.getenv(key_name))

if not API_KEYS:
    print("Error: No GEMINI API keys found in .env")
else:
    print(f"Loaded {len(API_KEYS)} GEMINI API keys.")

# Rate Limiting Key Manager
class KeyManager:
    def __init__(self, keys, min_interval=4.0):
        self.keys = keys
        self.min_interval = min_interval
        self.last_used = {key: 0 for key in keys}
        self.lock = threading.Lock()
        self.index = 0

    def get_key(self):
        with self.lock:
            key = self.keys[self.index]
            self.index = (self.index + 1) % len(self.keys)
            
            current_time = time.time()
            last_time = self.last_used[key]
            wait_time = 0
            
            if current_time - last_time < self.min_interval:
                wait_time = self.min_interval - (current_time - last_time)
            
            # Update last_used to the future time when it will be effectively used
            self.last_used[key] = current_time + wait_time
            
        if wait_time > 0:
            time.sleep(wait_time)
            
        return key

if API_KEYS:
    key_manager = KeyManager(API_KEYS, min_interval=5.0) # 5s interval to be safe
else:
    key_manager = None

# Configuration
PDF_DIR = "../pdfs"
DB_DIR = "./chroma_db"

if not os.path.exists(PDF_DIR):
    print(f"Warning: PDF directory {PDF_DIR} does not exist. Please check the path.")
else:
    print(f"PDF Directory: {os.path.abspath(PDF_DIR)}")

/home/rishabh/coding/pro/RAG/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 2 GEMINI API keys.
PDF Directory: /home/rishabh/coding/pro/RAG/pdfs


In [ ]:
valid_departments = [
    "ARTIFICIAL INTELLIGENCE AND DATA ENGINEERING",
    "ARCHITECTURE AND PLANNING",
    "CHEMICAL ENGINEERING",
    "CIVIL ENGINEERING",
    "COMPUTER SCIENCE AND ENGINEERING",
    "ELECTRICAL ENGINEERING",
    "ELECTRONICS AND COMMUNICATION ENGINEERING",
    "MECHANICAL ENGINEERING",
    "METALLURGICAL AND MATERIALS ENGINEERING",
    "CHEMISTRY",
    "MATHEMATICS",
    "PHYSICS",
    "MANAGEMENT STUDIES"
]

valid_centres = [
    "CENTRE FOR ENERGY AND ENVIRONMENT",
    "NATIONAL CENTRE FOR DISASTER MITIGATION AND MANAGEMENT",
    "MATERIALS RESEARCH CENTRE"
]

class DocumentMetadata(BaseModel):
    year: str = Field(description="The year (e.g., '2024'). If not found, 'Unknown'.")
    Dept: str = Field(description="Department or Centre name. Must be strictly one of the valid departments or centres listed in the prompt.")
    doc_type: str = Field(description="Type (e.g., 'Syllabus', 'Notice').")
    audience: str = Field(description="Target audience (e.g., 'UG', 'Faculty').")
    title: str = Field(description="Concise title.")
    summary: str = Field(description="1-sentence summary.")

parser = JsonOutputParser(pydantic_object=DocumentMetadata)

dept_str = json.dumps(valid_departments, indent=2)
centre_str = json.dumps(valid_centres, indent=2)

prompt_template_str = """Analyze the first page text from MNIT.
Extract the metadata fields as specified.

CRITICAL INSTRUCTION FOR 'Dept' FIELD:
You must strictly map the Department or Centre mentioned in the text to one of the exact names in the lists below.
for example if the text uses short forms (e.g., 'CS', 'CSE', 'Comp Sc'), map it to 'COMPUTER SCIENCE AND ENGINEERING'.
if it uses 'AIDE' or 'AI', map it to 'ARTIFICIAL INTELLIGENCE AND DATA ENGINEERING'.
If the department is not in the list, use 'Institute' if it concerns for the entire institute, or 'Mess' if it is related to the mess similary other, if nothign is relevent then 'Unknown'.

Valid Departments:
{valid_departments}

Valid Centres:
{valid_centres}

{format_instructions}
Text:
{text}
"""

prompt = PromptTemplate(
    template=prompt_template_str,
    input_variables=["text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions(),
        "valid_departments": dept_str,
        "valid_centres": centre_str
    },
)

def get_llm_chain(api_key):
    """Creates a chain with a specific API key."""
    llm = ChatGoogleGenerativeAI(
        model="gemma-3-27b-it", 
        temperature=0,
        google_api_key=api_key
    )
    return prompt | llm | parser

In [ ]:
def extract_metadata(text, retries=3):
    """Extracts metadata with key rotation and retry logic."""
    for attempt in range(retries):
        # Get key from manager (handles rate limiting)
        current_key = key_manager.get_key()
        chain = get_llm_chain(current_key)
        
        try:
            return chain.invoke({"text": text[:3000]})
        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                print(f"  Rate limit on key ...{current_key[-4:]}. Retrying...")
                time.sleep(2) 
            elif "503" in error_str:
                print(f"  Model overloaded. Retrying...")
                time.sleep(5)
            else:
                print(f"  Error: {e}")
                break
                
    # Fallback
    print("!!!!!failed!!!")
    return {
        "year": "Unknown",
        "Dept": "Unknown",
        "doc_type": "Unknown",
        "audience": "Unknown",
        "title": "Unknown",
        "summary": "Error extracting metadata"
    }

In [7]:
# List available models to verify access
print("Available models:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

Available models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025


In [ ]:
documents_to_index = []

def process_single_pdf(pdf_path):
    """Process a single PDF: Load, Extract Metadata, and Tag Pages."""
    try:
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        
        if not docs:
            return []

        # Extract metadata ONLY from the first page
        first_page_text = docs[0].page_content
        meta_data = extract_metadata(first_page_text)
        
        # Add file path to metadata
        meta_data["source"] = pdf_path
        
        # Apply this metadata to ALL pages
        for doc in docs:
            doc.metadata.update(meta_data)
            
        return docs
                
    except Exception as e:
        print(f"Failed to process {os.path.basename(pdf_path)}: {e}")
        return []

if not os.path.exists(PDF_DIR):
    print(f"Error: Directory {PDF_DIR} not found.")
else:
    pdf_files = glob.glob(os.path.join(PDF_DIR, "*.pdf"))
    total_files = len(pdf_files)
    print(f"Found {total_files} PDFs. Processing in parallel with {len(API_KEYS)} API keys...")

    # Use ThreadPoolExecutor for parallel processing
    # Set max_workers to len(API_KEYS) to match our key capacity
    max_workers = len(API_KEYS) 
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_pdf = {executor.submit(process_single_pdf, pdf): pdf for pdf in pdf_files}
        
        completed_count = 0
        for future in concurrent.futures.as_completed(future_to_pdf):
            pdf_path = future_to_pdf[future]
            try:
                docs = future.result()
                documents_to_index.extend(docs)
                completed_count += 1
                
                # Progress Log
                if completed_count % 5 == 0:
                    print(f"Progress: {completed_count}/{total_files} PDFs processed.")
                    
            except Exception as exc:
                print(f"{os.path.basename(pdf_path)} generated an exception: {exc}")

print(f"Total documents (pages) to index: {len(documents_to_index)}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Found 130 PDFs. Processing in parallel with 2 API keys...


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Progress: 5/130 PDFs processed.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


...

  Rate limit on key ...1aRA. Retrying with next key...
  Rate limit on key ...vS_w. Retrying with next key...
  Rate limit on key ...vS_w. Retrying with next key...


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


  Rate limit on key ...vS_w. Retrying with next key...


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Progress: 125/130 PDFs processed.
Progress: 130/130 PDFs processed.
Total documents (pages) to index: 1191
Progress: 130/130 PDFs processed.
Total documents (pages) to index: 1191


In [9]:
# --- 3. SPLIT DOCUMENTS ---
if documents_to_index:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(documents_to_index)
    print(f"Created {len(splits)} chunks from {len(documents_to_index)} pages.")

    # --- 4. EMBED AND SAVE ---
    print("Initializing Google Embeddings...")
    
    # 1. Use the modern 'text-embedding-004' model
    # 2. Let LangChain auto-detect the key from os.environ (safer)
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/text-embedding-004" 
    )

    # Create Vector DB
    if splits:
        vectorstore = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            persist_directory=DB_DIR
        )
        print(f"Successfully saved to {DB_DIR}")
else:
    print("No documents were processed successfully.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Created 2339 chunks from 1191 pages.
Initializing Google Embeddings...
Successfully saved to ./chroma_db
Successfully saved to ./chroma_db
